In [ ]:
pip install -U openai python-dotenv

In [ ]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"

client = OpenAI()

MODEL = "gpt-4o-mini"  # cheap + good

BANDS = [
    {"label": "Budget", "range": "< $20"},
    {"label": "Mid", "range": "$20–$50"},
    {"label": "Premium", "range": "$50–$100"},
    {"label": "Luxury", "range": "> $100"},
]


In [ ]:

def _call(messages, temperature=0.2):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=300,
    )
    return resp.choices[0].message.content.strip()

In [ ]:
def agent_prompt(role_name: str, role_bias: str, product: str, band_defs: str) -> list[dict]:
    system = (
        f"You are {role_name}, a pricing agent.\n"
        f"Your bias: strongly consider {role_bias} as the correct price band.\n"
        "You must still be honest and update if evidence suggests otherwise.\n"
        "Return JSON only: {\"band\": \"<label>\", \"confidence\": 0-1, \"reasons\": [..], \"questions\": [..]}.\n"
        "No markdown."
    )
    user = (
        f"Price bands:\n{band_defs}\n\n"
        f"Product description:\n{product}\n\n"
        "Pick the most likely band."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

In [ ]:

def rebut_prompt(role_name: str, product: str, band_defs: str, others: list[dict]) -> list[dict]:
    system = (
        f"You are {role_name}.\n"
        "You will rebut other agents. Be concise.\n"
        "Return JSON only: {\"keep_band\": \"<label>\", \"confidence\": 0-1, \"rebuttal\": \"...\"}.\n"
        "No markdown."
    )
    user = (
        f"Price bands:\n{band_defs}\n\n"
        f"Product:\n{product}\n\n"
        f"Other agents' proposals:\n{json.dumps(others, indent=2)}\n\n"
        "Rebut them and update your view if needed."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

In [ ]:
def moderator_prompt(product: str, band_defs: str, proposals: list[dict], rebuttals: list[dict]) -> list[dict]:
    system = (
        "You are the Moderator. Choose the final price band.\n"
        "Weigh consensus, evidence quality, and confidence.\n"
        "Return JSON only: {\"final_band\": \"<label>\", \"confidence\": 0-1, "
        "\"rationale\": \"...\", \"votes\": {\"Budget\": n, \"Mid\": n, \"Premium\": n, \"Luxury\": n}}.\n"
        "No markdown."
    )
    user = (
        f"Price bands:\n{band_defs}\n\n"
        f"Product:\n{product}\n\n"
        f"Proposals:\n{json.dumps(proposals, indent=2)}\n\n"
        f"Rebuttals:\n{json.dumps(rebuttals, indent=2)}\n\n"
        "Pick one final band."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

In [ ]:
def classify_with_debate(product: str) -> dict:
    band_defs = "\n".join([f"- {b['label']}: {b['range']}" for b in BANDS])

    agents = [
        ("Agent Budget", "Budget", "Budget"),
        ("Agent Mid", "Mid", "Mid"),
        ("Agent Premium", "Premium or Luxury", "Premium"),
    ]

    # Round 1: proposals
    proposals = []
    for name, bias, default_band in agents:
        out = _call(agent_prompt(name, bias, product, band_defs))
        try:
            j = json.loads(out)
        except json.JSONDecodeError:
            j = {"band": default_band, "confidence": 0.3, "reasons": ["(parse-fallback)"], "questions": []}
        proposals.append({"agent": name, **j})

    # Round 2: rebuttals
    rebuttals = []
    for p in proposals:
        name = p["agent"]
        out = _call(rebut_prompt(name, product, band_defs, proposals))
        try:
            j = json.loads(out)
        except json.JSONDecodeError:
            j = {"keep_band": p.get("band", "Mid"), "confidence": p.get("confidence", 0.3), "rebuttal": "(parse-fallback)"}
        rebuttals.append({"agent": name, **j})

         # Moderator decision
    out = _call(moderator_prompt(product, band_defs, proposals, rebuttals), temperature=0.0)
    result = json.loads(out)

    return {
        "product": product,
        "proposals": proposals,
        "rebuttals": rebuttals,
        "decision": result,
    }

In [ ]:
example = "Noise-cancelling over-ear Bluetooth headphones, 40-hour battery, carrying case, fast charging, premium build."
debate = classify_with_debate(example)
print(json.dumps(debate["decision"], indent=2))

In [ ]:
import gradio as gr
import json

def run_debate_ui(product_description: str):
    if not product_description or len(product_description.strip()) < 10:
        return "Please enter a longer product description.", "", "", ""

    debate = classify_with_debate(product_description.strip())

    decision = json.dumps(debate["decision"], indent=2)
    proposals = json.dumps(debate["proposals"], indent=2)
    rebuttals = json.dumps(debate["rebuttals"], indent=2)

    # A friendly summary line for the UI
    summary = (
        f"Final band: {debate['decision']['final_band']} "
        f"(confidence: {debate['decision']['confidence']:.2f})"
    )

    return summary, decision, proposals, rebuttals

with gr.Blocks(title="Multi-Agent Price Band Debate") as demo:
    gr.Markdown("# 🗣️ Multi-Agent Debate: Price Band Classifier")
    gr.Markdown(
        "Enter a product description. Three agents debate (Budget/Mid/Premium), "
        "then a Moderator selects the final band."
    )

    product = gr.Textbox(
        label="Product description",
        placeholder="e.g., Wireless noise-cancelling headphones, 40h battery, premium build...",
        lines=6,
    )
    btn = gr.Button("Run debate")

    summary = gr.Textbox(label="Result", interactive=False)
    decision = gr.Code(label="Moderator decision (JSON)", language="json")
    proposals = gr.Code(label="Agent proposals (JSON)", language="json")
    rebuttals = gr.Code(label="Agent rebuttals (JSON)", language="json")

    btn.click(
        fn=run_debate_ui,
        inputs=[product],
        outputs=[summary, decision, proposals, rebuttals],
    )

demo.launch(share=False)